# Calories Prediction — Data Visualization

## English

This notebook performs exploratory data visualization for the Calories Prediction project. It analyzes the distribution of the target variable, relationships between physical activity features, gender differences, and correlations between numerical variables.

## العربية

يهدف هذا الدفتر إلى تنفيذ التحليل البصري الاستكشافي لمشروع التنبؤ بالسعرات الحرارية. يقوم بتحليل توزيع المتغير المستهدف، والعلاقات بين خصائص النشاط البدني، والفروقات حسب الجنس، والارتباطات بين المتغيرات الرقمية.


## Visualization Objectives

### English

The visualization stage aims to:

- Understand the distribution of the target variable.
- Understand the distribution of numerical features.
- Identify possible outliers.
- Analyze relationships between features.
- Analyze relationships between features and Calories Burned.
- Compare Calories Burned by Gender.
- Study correlations between numerical variables.
- Identify potentially useful features for Machine Learning.
- Generate reusable figures for the project reports.

### العربية

مرحلة التصور البصري تهدف إلى:

- فهم توزيع المتغير المستهدف.
- فهم توزيع المتغيرات الرقمية.
- اكتشاف القيم المتطرفة المحتملة.
- تحليل العلاقات بين الخصائص.
- تحليل العلاقة بين الخصائص والسعرات المحروقة.
- مقارنة السعرات المحروقة حسب الجنس.
- دراسة الارتباطات بين المتغيرات الرقمية.
- تحديد الخصائص التي قد تكون مفيدة للتعلم الآلي.
- إنشاء صور قابلة لإعادة الاستخدام في تقارير المشروع.


## Imports and Configuration

### English

This section imports all the libraries required for data handling and visualization, configures a consistent plotting style, and defines the project paths so the notebook can locate the dataset and save figures reliably regardless of where it is executed from.

### العربية

يقوم هذا القسم باستيراد جميع المكتبات اللازمة لمعالجة البيانات والتصور البصري، وضبط نمط موحد للرسومات، وتحديد مسارات المشروع حتى يتمكن الدفتر من إيجاد ملف البيانات وحفظ الرسومات بشكل صحيح بغض النظر عن مكان التشغيل.


In [ ]:
# Core libraries
from pathlib import Path

import pandas as pd
import numpy as np

# Matplotlib configured with a non-GUI backend (Agg)
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns

# Used to display saved PNG files directly inside notebook cells
from IPython.display import display, Image

# ---------------------------------------------------------------------------
# Plotting style configuration
# ---------------------------------------------------------------------------
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# ---------------------------------------------------------------------------
# Robust project path resolution
# ---------------------------------------------------------------------------
# This notebook is expected to live inside: calories-prediction/notebooks/
# We resolve PROJECT_ROOT so that it always points to: calories-prediction/
# even if the notebook is executed from a different working directory.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dataset_clean.csv"
)

FIGURES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data file    : {DATA_FILE}")
print(f"Figures dir  : {FIGURES_DIR}")


Project root : c:\Users\ayman\OneDrive\Desktop\calories-prediction
Data file    : c:\Users\ayman\OneDrive\Desktop\calories-prediction\data\processed\dataset_clean.csv
Figures dir  : c:\Users\ayman\OneDrive\Desktop\calories-prediction\reports\figures


## Load Processed Dataset

### English

We load the cleaned dataset produced by the earlier data-cleaning stage of the project. This is the same dataset that will be used throughout the visualization notebook — no synthetic or manually entered data is used.

### العربية

نقوم بتحميل النسخة النظيفة من البيانات الناتجة عن مرحلة تنظيف البيانات السابقة في المشروع. هذه هي نفس البيانات التي سيتم استخدامها في جميع أنحاء دفتر التصور البصري، دون استخدام أي بيانات وهمية أو مدخلة يدويًا.


In [ ]:
# Load the cleaned dataset
df = pd.read_csv(r"C:\Users\ayman\OneDrive\Desktop\calories-prediction\data\raw\dataset.csv")

# Basic checks
print(f"Dataset loaded from: {DATA_FILE}")
print(f"Shape: {df.shape}")
print("\nColumn names:")
print(list(df.columns))

df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ayman\\OneDrive\\Desktop\\calories-prediction\\data\\processed\\dataset_clean.csv'

## Dataset Overview

### English

Before creating any visualization, it is important to understand the structure of the dataset: how many rows and columns it has, the data type of each column, and the basic statistical summary of the numerical columns. This helps us choose appropriate chart types and spot potential issues (e.g. unexpected data types) early.

### العربية

قبل إنشاء أي رسم بياني، من المهم فهم بنية البيانات: عدد الصفوف والأعمدة، ونوع البيانات لكل عمود، والملخص الإحصائي الأساسي للأعمدة الرقمية. يساعدنا ذلك في اختيار أنواع الرسومات المناسبة واكتشاف أي مشاكل محتملة (مثل أنواع بيانات غير متوقعة) في وقت مبكر.


In [ ]:
# Shape of the dataset
print("Shape (rows, columns):", df.shape)

# General info: column names, non-null counts, data types
print("\nDataset Info:")
df.info()


In [ ]:
# Data types of each column
print("Data types:\n")
print(df.dtypes)


In [ ]:
# Statistical summary of numerical columns
df.describe()


## Visualization Helper Function

### English

To keep the notebook consistent and avoid repeating code, we define a single reusable helper function `save_and_display_figure()`. It saves the current Matplotlib figure as a high-quality PNG (300 DPI) inside `reports/figures`, closes the figure to free memory, prints the exact saved path, and finally displays the saved PNG file directly inside the notebook output using `IPython.display.Image`. Every visualization in this notebook uses this same function.

### العربية

للحفاظ على اتساق الدفتر وتجنب تكرار الكود، نقوم بتعريف دالة مساعدة واحدة قابلة لإعادة الاستخدام باسم `save_and_display_figure()`. تقوم هذه الدالة بحفظ الرسم البياني الحالي كملف PNG عالي الجودة (بدقة 300 نقطة في البوصة) داخل مجلد `reports/figures`، ثم تغلق الرسم لتحرير الذاكرة، وتطبع المسار الفعلي الذي تم الحفظ فيه، وأخيرًا تعرض ملف PNG المحفوظ مباشرة داخل مخرجات الدفتر باستخدام `IPython.display.Image`. تستخدم جميع الرسومات في هذا الدفتر نفس هذه الدالة.


In [ ]:
def save_and_display_figure(filename):
    """
    Save the current figure and display the saved image
    inside the notebook.
    """

    file_path = FIGURES_DIR / filename

    plt.tight_layout()

    plt.savefig(
        file_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Figure saved: {file_path}")

    display(
        Image(
            filename=str(file_path)
        )
    )


## Calories Burned Distribution

### English

We start with the target variable, `Calories Burned`, using a histogram combined with a KDE (density) curve to visualize its overall shape.

### العربية

نبدأ بالمتغير المستهدف `Calories Burned`، باستخدام مخطط تكراري (Histogram) مدمج مع منحنى الكثافة (KDE) لتصور شكله العام.


In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="Calories Burned",
    kde=True,
    color="#d6537d",
    bins=25
)

plt.title("Calories Burned Distribution")
plt.xlabel("Calories Burned")
plt.ylabel("Frequency")

save_and_display_figure("calories_burned_distribution.png")


### Interpretation

**English:**
The distribution of `Calories Burned` is spread broadly across the observed range, without a single sharp peak. Several bars reach similar heights across the range, suggesting the values are relatively spread out rather than tightly concentrated around one value, and the KDE curve is fairly flat with only mild bumps. No extreme isolated bars are visible, so there is no strong sign of unusual outlier values in this variable.

**العربي:**
يظهر توزيع متغير `Calories Burned` انتشارًا واسعًا عبر النطاق الملحوظ، دون وجود قمة حادة واحدة. تصل عدة أعمدة إلى ارتفاعات متقاربة عبر النطاق، مما يشير إلى أن القيم منتشرة نسبيًا بدلاً من أن تتركز حول قيمة واحدة، كما أن منحنى الكثافة يبدو مسطحًا نسبيًا مع تموجات بسيطة فقط. لا تظهر أي أعمدة متطرفة معزولة بشكل واضح، لذلك لا توجد إشارة قوية لوجود قيم متطرفة غير اعتيادية في هذا المتغير.


## Numerical Feature Distributions

### English

In this section we create a separate histogram with a KDE curve for each numerical feature in the dataset. Each figure is saved individually so it remains readable, rather than combining all features into a single crowded plot.

### العربية

في هذا القسم، نقوم بإنشاء مخطط تكراري منفصل مع منحنى كثافة لكل خاصية رقمية في البيانات. يتم حفظ كل رسم بشكل منفصل حتى يبقى واضحًا للقراءة، بدلاً من دمج جميع الخصائص في رسم واحد مزدحم.


In [ ]:
numerical_features = [
    "Age",
    "Height(cm)",
    "Weight(kg)",
    "BMI",
    "Running Time(min)",
    "Running Speed(km/h)",
    "Distance(km)",
    "Average Heart Rate",
    "Calories Burned",
]

filename_map = {
    "Age": "distribution_age.png",
    "Height(cm)": "distribution_height.png",
    "Weight(kg)": "distribution_weight.png",
    "BMI": "distribution_bmi.png",
    "Running Time(min)": "distribution_running_time.png",
    "Running Speed(km/h)": "distribution_running_speed.png",
    "Distance(km)": "distribution_distance.png",
    "Average Heart Rate": "distribution_average_heart_rate.png",
    "Calories Burned": "distribution_calories_burned.png",
}

for feature in numerical_features:
    plt.figure(figsize=(10, 6))

    sns.histplot(
        data=df,
        x=feature,
        kde=True,
        color="#4C72B0",
        bins=25
    )

    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")

    save_and_display_figure(filename_map[feature])


### Interpretation

**English:**
- **Age:** spread across roughly 18–65 years, with a tall bar around 18–20, a dip around 30, and the tallest bar around 48–50. The shape is irregular / multi-modal rather than a single smooth peak, with a mild right-hand taper toward the older ages.
- **Height(cm):** spread across roughly 150–199 cm with several local peaks (near 151, 170, and 183 cm) rather than one dominant center, and no isolated extreme bars.
- **Weight(kg):** spread across roughly 50–99 kg with local peaks near 60, 77, and 95 kg, again a multi-modal rather than single-peak shape, without isolated extreme bars.
- **BMI:** spread across roughly 18–30 with a broad concentration between about 23 and 28, including two adjacent tall bars around 24, and smaller counts at the low end (18–20); the KDE curve shows a fairly rounded single broad hump in that 22–28 region.
- **Running Time(min):** spread across roughly 20–120 minutes, without one sharp peak — several bars in the 65–100 minute region reach similarly tall heights, indicating a broad plateau rather than a narrow concentration.
- **Running Speed(km/h):** spread across roughly 7–15 km/h, with the tallest bars around 10–11 km/h and again around 13.5 km/h, giving the shape two noticeable high points rather than one.
- **Distance(km):** spread across roughly 3–15 km, with a tall bar at the low end (around 3 km) and another cluster of tall bars near 13 km, and comparatively lower bars in the 10–11 km region — a two-sided rather than centrally concentrated shape.
- **Average Heart Rate:** spread across roughly 120–180, without a single sharp peak; several bars around 130, 139, and 152 reach similar heights, indicating a broad, fairly even spread.
- **Calories Burned:** as shown earlier, spread broadly across 200–1000 with a tall bar near 500 and no isolated extreme bars.

Overall, none of the numerical features in this dataset show a single narrow concentrated peak; all of them are relatively spread out across their range and several are visibly multi-modal (more than one local peak). No feature shows an isolated bar far from the rest of the distribution, so there is no strong visual sign of extreme outlier values at the histogram level.

**العربي:**
- **العمر (Age):** ينتشر تقريبًا بين 18 و65 عامًا، مع عمود مرتفع بين 18-20 عامًا، وانخفاض حول سن 30، وأعلى عمود حول 48-50 عامًا. الشكل غير منتظم / متعدد المنوالات بدلاً من قمة واحدة سلسة، مع تناقص خفيف نحو الأعمار الأكبر.
- **الطول (Height(cm)):** ينتشر تقريبًا بين 150 و199 سم مع عدة قمم محلية (قرب 151 و170 و183 سم) بدلاً من مركز واحد مهيمن، دون أعمدة متطرفة معزولة.
- **الوزن (Weight(kg)):** ينتشر تقريبًا بين 50 و99 كجم مع قمم محلية قرب 60 و77 و95 كجم، وهو شكل متعدد المنوالات أيضًا دون أعمدة متطرفة معزولة.
- **مؤشر كتلة الجسم (BMI):** ينتشر تقريبًا بين 18 و30 مع تركز واسع بين 23 و28 تقريبًا، بما في ذلك عمودان متجاوران مرتفعان قرب 24، وأعداد أقل عند الطرف المنخفض (18-20)؛ ويُظهر منحنى الكثافة قمة واحدة واسعة ومقوسة في منطقة 22-28 تقريبًا.
- **زمن الجري (Running Time(min)):** ينتشر تقريبًا بين 20 و120 دقيقة، دون قمة حادة واحدة — تصل عدة أعمدة في نطاق 65-100 دقيقة إلى ارتفاعات متقاربة، مما يشير إلى هضبة واسعة بدلاً من تركز ضيق.
- **سرعة الجري (Running Speed(km/h)):** ينتشر تقريبًا بين 7 و15 كم/سا، مع أعلى الأعمدة قرب 10-11 كم/سا ومرة أخرى قرب 13.5 كم/سا، مما يعطي الشكل نقطتين مرتفعتين ملحوظتين بدلاً من واحدة.
- **المسافة (Distance(km)):** تنتشر تقريبًا بين 3 و15 كم، مع عمود مرتفع عند الطرف المنخفض (قرب 3 كم) ومجموعة أخرى من الأعمدة المرتفعة قرب 13 كم، وأعمدة أقل نسبيًا في نطاق 10-11 كم — شكل ذو طرفين بدلاً من تركز مركزي.
- **متوسط معدل ضربات القلب (Average Heart Rate):** ينتشر تقريبًا بين 120 و180، دون قمة حادة واحدة؛ تصل عدة أعمدة قرب 130 و139 و152 إلى ارتفاعات متقاربة، مما يشير إلى انتشار واسع ومتوازن نسبيًا.
- **السعرات المحروقة (Calories Burned):** كما ذُكر سابقًا، تنتشر بشكل واسع بين 200 و1000 مع عمود مرتفع قرب 500 ودون أعمدة متطرفة معزولة.

بشكل عام، لا تُظهر أي من الخصائص الرقمية في هذه البيانات قمة واحدة ضيقة ومركزة؛ فجميعها منتشرة نسبيًا عبر نطاقها، وبعضها متعدد المنوالات بشكل واضح (أكثر من قمة محلية واحدة). لا تُظهر أي خاصية عمودًا معزولًا بعيدًا عن بقية التوزيع، لذلك لا توجد إشارة بصرية قوية لوجود قيم متطرفة شديدة على مستوى المخطط التكراري.


## Numerical Feature Boxplots

### English

Boxplots are useful for quickly identifying the median, the spread (interquartile range), and potential outliers of each numerical feature. We generate a separate boxplot for each numerical feature.

### العربية

تُعد مخططات الصندوق (Boxplots) مفيدة للتعرف بسرعة على الوسيط ومدى الانتشار (المدى الربيعي) والقيم المتطرفة المحتملة لكل خاصية رقمية. نقوم بإنشاء مخطط صندوق منفصل لكل خاصية رقمية.


In [ ]:
boxplot_filename_map = {
    "Age": "boxplot_age.png",
    "Height(cm)": "boxplot_height.png",
    "Weight(kg)": "boxplot_weight.png",
    "BMI": "boxplot_bmi.png",
    "Running Time(min)": "boxplot_running_time.png",
    "Running Speed(km/h)": "boxplot_running_speed.png",
    "Distance(km)": "boxplot_distance.png",
    "Average Heart Rate": "boxplot_average_heart_rate.png",
    "Calories Burned": "boxplot_calories_burned.png",
}

for feature in numerical_features:
    plt.figure(figsize=(10, 6))

    sns.boxplot(
        data=df,
        y=feature,
        color="#DD8452"
    )

    plt.title(f"Boxplot of {feature}")
    plt.ylabel(feature)

    save_and_display_figure(boxplot_filename_map[feature])


### Interpretation

**English:**
- **Age:** median around 44, box (IQR) roughly 30–52, whiskers extending to about 18–65. No points beyond the whiskers.
- **Height(cm):** median around 175, box roughly 163–186, whiskers to about 150–199. No points beyond the whiskers.
- **Weight(kg):** median around 74–75, box roughly 62–87, whiskers to about 50–99. No points beyond the whiskers.
- **BMI:** median around 24.5, box roughly 21.6–27, whiskers to about 18.5–30. No points beyond the whiskers.
- **Running Time(min):** median around 74, box roughly 56–95, whiskers to about 30–120. No points beyond the whiskers.
- **Running Speed(km/h):** median around 11.3, box roughly 9.8–13.1, whiskers to about 7–15. No points beyond the whiskers.
- **Distance(km):** median around 8.8, box roughly 5.5–12.6, whiskers to about 3–15. No points beyond the whiskers.
- **Average Heart Rate:** median around 150, box roughly 135–164, whiskers to about 121–179. No points beyond the whiskers.
- **Calories Burned:** median around 555, box roughly 395–790, whiskers to about 200–1000. No points beyond the whiskers.

Across all nine features, none of the boxplots show any individual point plotted beyond the whiskers, so there is no visual evidence of outliers by the standard boxplot rule in this dataset. Every feature's box and whiskers span a wide portion of its overall range, which is consistent with the broad, spread-out distributions seen in the histograms rather than a tightly clustered set of values.

**العربي:**
- **العمر:** الوسيط قرابة 44، ويغطي الصندوق (المدى الربيعي) تقريبًا 30-52، وتمتد الشعيرات إلى حوالي 18-65. لا توجد نقاط خارج الشعيرات.
- **الطول (سم):** الوسيط قرابة 175، ويغطي الصندوق تقريبًا 163-186، وتمتد الشعيرات إلى حوالي 150-199. لا توجد نقاط خارج الشعيرات.
- **الوزن (كجم):** الوسيط قرابة 74-75، ويغطي الصندوق تقريبًا 62-87، وتمتد الشعيرات إلى حوالي 50-99. لا توجد نقاط خارج الشعيرات.
- **مؤشر كتلة الجسم:** الوسيط قرابة 24.5، ويغطي الصندوق تقريبًا 21.6-27، وتمتد الشعيرات إلى حوالي 18.5-30. لا توجد نقاط خارج الشعيرات.
- **زمن الجري (دقيقة):** الوسيط قرابة 74، ويغطي الصندوق تقريبًا 56-95، وتمتد الشعيرات إلى حوالي 30-120. لا توجد نقاط خارج الشعيرات.
- **سرعة الجري (كم/سا):** الوسيط قرابة 11.3، ويغطي الصندوق تقريبًا 9.8-13.1، وتمتد الشعيرات إلى حوالي 7-15. لا توجد نقاط خارج الشعيرات.
- **المسافة (كم):** الوسيط قرابة 8.8، ويغطي الصندوق تقريبًا 5.5-12.6، وتمتد الشعيرات إلى حوالي 3-15. لا توجد نقاط خارج الشعيرات.
- **متوسط معدل ضربات القلب:** الوسيط قرابة 150، ويغطي الصندوق تقريبًا 135-164، وتمتد الشعيرات إلى حوالي 121-179. لا توجد نقاط خارج الشعيرات.
- **السعرات المحروقة:** الوسيط قرابة 555، ويغطي الصندوق تقريبًا 395-790، وتمتد الشعيرات إلى حوالي 200-1000. لا توجد نقاط خارج الشعيرات.

عبر جميع الخصائص التسع، لا يُظهر أي من مخططات الصندوق أي نقطة فردية خارج الشعيرات، لذلك لا يوجد دليل بصري على وجود قيم متطرفة وفق قاعدة مخطط الصندوق القياسية في هذه البيانات. يغطي صندوق وشعيرات كل خاصية جزءًا واسعًا من نطاقها الكلي، وهو ما يتوافق مع التوزيعات الواسعة والمنتشرة التي شوهدت في المخططات التكرارية بدلاً من مجموعة قيم متمركزة بشكل ضيق.


## Gender Distribution

### English

We visualize the number of observations for each gender category using a count plot.

### العربية

نقوم بتصور عدد السجلات لكل فئة من فئات الجنس باستخدام مخطط عد (Count Plot).


In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=df,
    x="Gender",
    color="#4C72B0"
)

plt.title("Distribution of Gender")
plt.xlabel("Gender")
plt.ylabel("Count")

save_and_display_figure("gender_distribution.png")

print(df["Gender"].value_counts())


### Interpretation

**English:**
The dataset contains a balanced number of Male and Female observations. Both bars reach the same height, indicating the two gender groups are represented in equal (or near-equal) proportions, which is favorable for a fair comparison between groups later in the analysis.

**العربي:**
تحتوي البيانات على عدد متوازن من السجلات لفئتي الذكور والإناث. يصل كلا العمودين إلى نفس الارتفاع تقريبًا، مما يشير إلى أن الفئتين ممثلتان بنسب متساوية (أو شبه متساوية)، وهو أمر مناسب لإجراء مقارنة عادلة بين الفئتين لاحقًا في التحليل.


## Calories Burned by Gender

### English

We compare the distribution of `Calories Burned` between Male and Female using a boxplot.

### العربية

نقارن توزيع `Calories Burned` بين الذكور والإناث باستخدام مخطط صندوق.


In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="Gender",
    y="Calories Burned",
    color="#DD8452"
)

plt.title("Calories Burned by Gender")
plt.xlabel("Gender")
plt.ylabel("Calories Burned")

save_and_display_figure("calories_by_gender.png")

print(df.groupby("Gender")["Calories Burned"].describe())


### Interpretation

**English:**
The visualization shows an observed difference between the Male and Female groups: the Male group's median appears somewhat higher than the Female group's median, and both boxes cover a wide and largely overlapping range from low to high calorie values. The overlap between the two boxes is substantial, so while a mild difference in central tendency is observed, the two groups are not clearly separated. This should not be interpreted as gender causing a difference in calories burned — it is simply an observed pattern in this dataset.

**العربي:**
يُظهر الرسم البياني فرقًا ملحوظًا بين مجموعتي الذكور والإناث: يبدو وسيط مجموعة الذكور أعلى قليلاً من وسيط مجموعة الإناث، ويغطي كلا الصندوقين نطاقًا واسعًا ومتداخلًا إلى حد كبير من القيم المنخفضة إلى المرتفعة للسعرات المحروقة. التداخل بين الصندوقين كبير، لذا وعلى الرغم من ملاحظة فرق بسيط في النزعة المركزية، فإن المجموعتين غير منفصلتين بشكل واضح. لا ينبغي تفسير ذلك على أن الجنس هو سبب الفرق في السعرات المحروقة — إنه مجرد نمط ملحوظ في هذه البيانات.


## Calories vs Running Time

### English

We examine the relationship between `Running Time(min)` and `Calories Burned` using a scatter plot, and quantify it with the Pearson correlation coefficient.

### العربية

نفحص العلاقة بين `Running Time(min)` و `Calories Burned` باستخدام مخطط انتشار، ونقيس هذه العلاقة باستخدام معامل ارتباط بيرسون.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Running Time(min)",
    y="Calories Burned",
    color="#4C72B0",
    alpha=0.6
)

plt.title("Calories Burned vs Running Time")
plt.xlabel("Running Time(min)")
plt.ylabel("Calories Burned")

corr_running_time = df["Running Time(min)"].corr(df["Calories Burned"])
print(f"Pearson correlation (Running Time vs Calories Burned): {corr_running_time:.2f}")

save_and_display_figure("calories_vs_running_time.png")


### Interpretation

**English:**
The scatter plot together with the printed Pearson correlation value determines the strength and direction of the relationship. A correlation close to 0 indicates a weak or unclear linear relationship, while values approaching +1 or -1 indicate a strong positive or negative relationship respectively. The printed value above should be used as the definitive measure rather than relying on visual impression alone.

**العربي:**
يُحدد مخطط الانتشار مع قيمة معامل ارتباط بيرسون المطبوعة قوة واتجاه العلاقة. القيمة القريبة من الصفر تشير إلى علاقة خطية ضعيفة أو غير واضحة، بينما القيم القريبة من +1 أو -1 تشير إلى علاقة قوية موجبة أو سالبة على التوالي. يجب الاعتماد على القيمة المطبوعة أعلاه كمقياس نهائي بدلاً من الاعتماد فقط على الانطباع البصري.


## Calories vs Distance

### English

We examine the relationship between `Distance(km)` and `Calories Burned`.

### العربية

نفحص العلاقة بين `Distance(km)` و `Calories Burned`.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Distance(km)",
    y="Calories Burned",
    color="#4C72B0",
    alpha=0.6
)

plt.title("Calories Burned vs Distance")
plt.xlabel("Distance(km)")
plt.ylabel("Calories Burned")

corr_distance = df["Distance(km)"].corr(df["Calories Burned"])
print(f"Pearson correlation (Distance vs Calories Burned): {corr_distance:.2f}")

save_and_display_figure("calories_vs_distance.png")


### Interpretation

**English:**
Use the printed Pearson correlation value together with the scatter pattern to judge whether Distance shows a meaningful linear relationship with Calories Burned in this dataset, or whether the points appear scattered with no clear trend.

**العربي:**
استخدم قيمة معامل ارتباط بيرسون المطبوعة إلى جانب نمط الانتشار للحكم على ما إذا كانت المسافة تُظهر علاقة خطية ذات دلالة مع السعرات المحروقة في هذه البيانات، أو أن النقاط تبدو متناثرة دون اتجاه واضح.


## Calories vs Average Heart Rate

### English

We examine the relationship between `Average Heart Rate` and `Calories Burned`.

### العربية

نفحص العلاقة بين `Average Heart Rate` و `Calories Burned`.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Average Heart Rate",
    y="Calories Burned",
    color="#4C72B0",
    alpha=0.6
)

plt.title("Calories Burned vs Average Heart Rate")
plt.xlabel("Average Heart Rate")
plt.ylabel("Calories Burned")

corr_heart_rate = df["Average Heart Rate"].corr(df["Calories Burned"])
print(f"Pearson correlation (Average Heart Rate vs Calories Burned): {corr_heart_rate:.2f}")

save_and_display_figure("calories_vs_heart_rate.png")


### Interpretation

**English:**
Compare the printed correlation value against the visual scatter to determine whether higher average heart rate values in this dataset tend to accompany higher (or lower) calories burned, or whether no clear pattern is present.

**العربي:**
قارن قيمة معامل الارتباط المطبوعة مع نمط الانتشار البصري لتحديد ما إذا كانت القيم الأعلى لمتوسط معدل ضربات القلب في هذه البيانات ترافقها قيم أعلى (أو أقل) من السعرات المحروقة، أو أنه لا يوجد نمط واضح.


## Calories vs Weight

### English

We examine the relationship between `Weight(kg)` and `Calories Burned`.

### العربية

نفحص العلاقة بين `Weight(kg)` و `Calories Burned`.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Weight(kg)",
    y="Calories Burned",
    color="#4C72B0",
    alpha=0.6
)

plt.title("Calories Burned vs Weight")
plt.xlabel("Weight(kg)")
plt.ylabel("Calories Burned")

corr_weight = df["Weight(kg)"].corr(df["Calories Burned"])
print(f"Pearson correlation (Weight vs Calories Burned): {corr_weight:.2f}")

save_and_display_figure("calories_vs_weight.png")


### Interpretation

**English:**
Review the printed correlation value to judge the strength and direction of the linear relationship between body weight and calories burned observed in this dataset.

**العربي:**
راجع قيمة معامل الارتباط المطبوعة للحكم على قوة واتجاه العلاقة الخطية بين وزن الجسم والسعرات المحروقة الملاحظة في هذه البيانات.


## Calories vs Age

### English

We examine the relationship between `Age` and `Calories Burned`.

### العربية

نفحص العلاقة بين `Age` و `Calories Burned`.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Age",
    y="Calories Burned",
    color="#4C72B0",
    alpha=0.6
)

plt.title("Calories Burned vs Age")
plt.xlabel("Age")
plt.ylabel("Calories Burned")

corr_age = df["Age"].corr(df["Calories Burned"])
print(f"Pearson correlation (Age vs Calories Burned): {corr_age:.2f}")

save_and_display_figure("calories_vs_age.png")


### Interpretation

**English:**
Review the printed correlation value to judge whether Age shows a meaningful linear relationship with Calories Burned in this dataset.

**العربي:**
راجع قيمة معامل الارتباط المطبوعة للحكم على ما إذا كان العمر يُظهر علاقة خطية ذات دلالة مع السعرات المحروقة في هذه البيانات.


## Running Time vs Distance

### English

We examine the relationship between `Running Time(min)` and `Distance(km)`.

### العربية

نفحص العلاقة بين `Running Time(min)` و `Distance(km)`.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Running Time(min)",
    y="Distance(km)",
    color="#55A868",
    alpha=0.6
)

plt.title("Running Time vs Distance")
plt.xlabel("Running Time(min)")
plt.ylabel("Distance(km)")

corr_time_distance = df["Running Time(min)"].corr(df["Distance(km)"])
print(f"Pearson correlation (Running Time vs Distance): {corr_time_distance:.2f}")

save_and_display_figure("running_time_vs_distance.png")


### Interpretation

**English:**
Intuitively, one might expect running time and distance to be positively related. Compare this expectation with the printed Pearson correlation and the scatter pattern actually observed in this dataset before drawing any conclusion.

**العربي:**
بشكل بديهي، قد يتوقع المرء أن يكون هناك ارتباط موجب بين زمن الجري والمسافة. قارن هذا التوقع مع قيمة معامل ارتباط بيرسون المطبوعة ونمط الانتشار الفعلي الملاحظ في هذه البيانات قبل استخلاص أي استنتاج.


## Running Speed vs Distance

### English

We examine the relationship between `Running Speed(km/h)` and `Distance(km)`.

### العربية

نفحص العلاقة بين `Running Speed(km/h)` و `Distance(km)`.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Running Speed(km/h)",
    y="Distance(km)",
    color="#55A868",
    alpha=0.6
)

plt.title("Running Speed vs Distance")
plt.xlabel("Running Speed(km/h)")
plt.ylabel("Distance(km)")

corr_speed_distance = df["Running Speed(km/h)"].corr(df["Distance(km)"])
print(f"Pearson correlation (Running Speed vs Distance): {corr_speed_distance:.2f}")

save_and_display_figure("running_speed_vs_distance.png")


### Interpretation

**English:**
Review the printed correlation value together with the scatter plot to judge whether running speed and distance are related in this dataset, or whether the points appear largely scattered.

**العربي:**
راجع قيمة معامل الارتباط المطبوعة إلى جانب مخطط الانتشار للحكم على ما إذا كانت سرعة الجري والمسافة مرتبطتين في هذه البيانات، أو أن النقاط تبدو متناثرة إلى حد كبير.


## Correlation Heatmap

### English

We compute the Pearson correlation matrix for all numerical columns and visualize it as an annotated heatmap. This provides a compact overview of all pairwise linear relationships in the dataset at once.

### العربية

نقوم بحساب مصفوفة ارتباط بيرسون لجميع الأعمدة الرقمية، ونعرضها كخريطة حرارية موضحة بالقيم. يوفر هذا نظرة شاملة ومختصرة على جميع العلاقات الخطية الثنائية في البيانات دفعة واحدة.


In [ ]:
correlation_matrix = df.select_dtypes(include="number").corr()

plt.figure(figsize=(12, 9))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.title("Correlation Heatmap")

save_and_display_figure("correlation_heatmap.png")

correlation_matrix


### Interpretation

**English:**
Inspect the heatmap for the strongest positive correlations (values closest to +1, excluding the diagonal), the strongest negative correlations (values closest to -1), and pairs with correlations near 0, which indicate weak or negligible linear relationships. Pay particular attention to the `Calories Burned` row/column to see which features are most and least linearly associated with the target variable. A correlation value, no matter how strong, only reflects a linear statistical association — it does not by itself prove that one variable causes another.

**العربي:**
افحص الخريطة الحرارية لتحديد أقوى الارتباطات الموجبة (القيم الأقرب إلى +1، باستثناء القطر الرئيسي)، وأقوى الارتباطات السالبة (القيم الأقرب إلى -1)، والأزواج ذات الارتباط القريب من الصفر والتي تشير إلى علاقة خطية ضعيفة أو لا تُذكر. انتبه بشكل خاص لصف/عمود `Calories Burned` لمعرفة الخصائص الأكثر والأقل ارتباطًا خطيًا بالمتغير المستهدف. قيمة الارتباط، مهما كانت قوتها، تعكس فقط علاقة إحصائية خطية — ولا تثبت بحد ذاتها أن أحد المتغيرات يسبب الآخر.


## Feature Correlations with Calories Burned

### English

We isolate the correlation of every numerical feature with the target variable `Calories Burned`, sort the features by correlation strength, and visualize them as a horizontal bar chart for easy comparison.

### العربية

نقوم بعزل ارتباط كل خاصية رقمية مع المتغير المستهدف `Calories Burned`، ثم ترتيب الخصائص حسب قوة الارتباط، وتصورها كمخطط أعمدة أفقي لتسهيل المقارنة.


In [ ]:
target_correlations = (
    correlation_matrix["Calories Burned"]
    .drop("Calories Burned")
    .sort_values()
)

plt.figure(figsize=(10, 7))

colors = ["#C44E52" if v < 0 else "#4C72B0" for v in target_correlations.values]

plt.barh(
    target_correlations.index,
    target_correlations.values,
    color=colors
)

plt.title("Feature Correlations with Calories Burned")
plt.xlabel("Pearson Correlation")
plt.ylabel("Feature")
plt.axvline(0, color="black", linewidth=0.8)

save_and_display_figure("target_correlations.png")

correlation_table = pd.DataFrame({
    "Feature": target_correlations.index,
    "Correlation": target_correlations.values,
    "Absolute Correlation": target_correlations.abs().values
}).sort_values("Absolute Correlation", ascending=False).reset_index(drop=True)

print(correlation_table)


### Interpretation

**English:**
The bar chart ranks all features by their correlation with `Calories Burned`. Features with bars extending furthest from zero (in either direction) have the strongest linear association with the target, while features with bars very close to zero show little to no linear relationship. The printed table gives the exact values, sorted by absolute correlation, which is useful for later feature-selection discussions in the Machine Learning stage of the project.

**العربي:**
يرتب المخطط الشريطي جميع الخصائص حسب ارتباطها بـ `Calories Burned`. الخصائص التي تمتد أعمدتها لأبعد مسافة عن الصفر (في أي اتجاه) لها أقوى ارتباط خطي بالمتغير المستهدف، بينما الخصائص التي تكون أعمدتها قريبة جدًا من الصفر تُظهر علاقة خطية ضعيفة أو معدومة. يوفر الجدول المطبوع القيم الدقيقة، مرتبة حسب القيمة المطلقة للارتباط، وهو مفيد للنقاشات اللاحقة حول اختيار الخصائص في مرحلة التعلم الآلي من المشروع.


## Pair Plot

### English

A pair plot shows pairwise scatter plots between every pair of numerical variables, along with the univariate distribution of each variable on the diagonal. Because this dataset contains several numerical columns, we keep the plot readable by using small point sizes and a light transparency.

### العربية

يعرض مخطط Pair Plot مخططات انتشار ثنائية بين كل زوج من المتغيرات الرقمية، إلى جانب توزيع كل متغير بمفرده على القطر الرئيسي. نظرًا لاحتواء هذه البيانات على عدة أعمدة رقمية، نحافظ على وضوح الرسم باستخدام نقاط صغيرة الحجم وشفافية خفيفة.


In [ ]:
pairplot_features = numerical_features + ["Gender"]

pairplot_grid = sns.pairplot(
    df[pairplot_features],
    hue="Gender",
    plot_kws={"alpha": 0.5, "s": 20},
    diag_kind="kde",
    corner=True
)

pairplot_grid.fig.suptitle("Pair Plot of Numerical Features", y=1.02)

file_path = FIGURES_DIR / "pairplot_numerical_features.png"

pairplot_grid.savefig(
    file_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(pairplot_grid.fig)

print(f"Figure saved: {file_path}")

display(Image(filename=str(file_path)))


### English

The pair plot provides a broad overview of pairwise relationships and distributions.

### العربية

يوفر مخطط Pair Plot نظرة شاملة على العلاقات بين المتغيرات الرقمية وتوزيعاتها.


## Calories Distribution by Gender (Violin Plot)

### English

A violin plot combines a boxplot with a KDE, showing the full shape of the `Calories Burned` distribution for each gender group.

### العربية

يجمع مخطط الكمان (Violin Plot) بين مخطط الصندوق ومنحنى الكثافة، ويوضح الشكل الكامل لتوزيع `Calories Burned` لكل فئة من فئات الجنس.


In [ ]:
plt.figure(figsize=(10, 6))

sns.violinplot(
    data=df,
    x="Gender",
    y="Calories Burned",
    color="#8172B2"
)

plt.title("Calories Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("Calories Burned")

save_and_display_figure("violin_calories_by_gender.png")


### Interpretation

**English:**
The width of each violin at a given calories level reflects how many observations fall around that level, showing the density shape rather than only the median and quartiles as a plain boxplot would. In this dataset, both the Male and Female violins run across the full 200–1000 range, and each violin is visibly wider in more than one place rather than having a single bulge — the Male shape shows extra width around the 700–800 region as well as lower down, and the Female shape shows a wider body concentrated more toward the lower-middle part of the range (roughly 400–500), narrowing somewhat above that. The Male median (marked inside the violin) sits a little higher than the Female median, matching the small difference already seen in the boxplot comparison, but the two groups' overall shapes overlap across nearly the whole range rather than being cleanly separated.

**العربي:**
يعكس عرض كل شكل كمان عند مستوى معين من السعرات عدد الملاحظات المتمركزة حول ذلك المستوى، مما يوضح شكل الكثافة بدلاً من الاكتفاء بعرض الوسيط والربيعيات كما في مخطط الصندوق العادي. في هذه البيانات، يمتد شكلا الكمان الخاصان بالذكور والإناث عبر النطاق الكامل من 200 إلى 1000، ويظهر كل شكل اتساعًا ملحوظًا في أكثر من موضع بدلاً من انتفاخ واحد فقط — فشكل الذكور يُظهر اتساعًا إضافيًا في نطاق 700-800 وأيضًا في الجزء المنخفض، بينما يُظهر شكل الإناث جسمًا أوسع يتركز أكثر نحو الجزء المتوسط المنخفض من النطاق (حوالي 400-500) ثم يضيق قليلًا فوق ذلك. يقع وسيط الذكور (المُشار إليه داخل شكل الكمان) أعلى قليلًا من وسيط الإناث، وهو ما يتوافق مع الفرق الصغير الذي ظهر سابقًا في مقارنة مخطط الصندوق، إلا أن الشكلين العامين للفئتين متداخلان عبر معظم النطاق بدلاً من أن يكونا منفصلين بوضوح.


## Calories KDE Distribution

### English

We create a standalone KDE (kernel density estimate) plot for `Calories Burned` to focus purely on its smoothed density shape.

### العربية

نقوم بإنشاء مخطط كثافة (KDE) مستقل لمتغير `Calories Burned` للتركيز فقط على شكل الكثافة المُنعّم لتوزيعه.


In [ ]:
plt.figure(figsize=(10, 6))

sns.kdeplot(
    data=df,
    x="Calories Burned",
    fill=True,
    color="#C44E52"
)

plt.title("KDE of Calories Burned")
plt.xlabel("Calories Burned")
plt.ylabel("Density")

save_and_display_figure("kde_calories_burned.png")


### Interpretation

**English:**
The KDE curve smooths out the histogram bars into a continuous density estimate. For `Calories Burned` in this dataset, the curve is clearly bimodal: it rises to a first, taller peak around the 400–500 range, dips to a local low around 650, then rises again to a second, slightly lower peak around 800, before tapering off toward 1000. This two-peak shape means the data is not simply concentrated around one typical value — there appear to be two broad clusters of observations (a lower-calorie group and a higher-calorie group) rather than a single symmetric center.

**العربي:**
ينعّم منحنى الكثافة أعمدة المخطط التكراري إلى تقدير كثافة مستمر. بالنسبة لمتغير `Calories Burned` في هذه البيانات، يظهر المنحنى بوضوح ثنائي القمة: يرتفع إلى قمة أولى وأعلى في نطاق 400-500 تقريبًا، ثم ينخفض إلى نقطة منخفضة محلية قرب 650، ثم يرتفع مرة أخرى إلى قمة ثانية أقل ارتفاعًا قليلًا قرب 800، قبل أن يتناقص تدريجيًا نحو 1000. يعني هذا الشكل ثنائي القمة أن البيانات ليست مجرد متمركزة حول قيمة نموذجية واحدة — بل يبدو أن هناك مجموعتين واسعتين من الملاحظات (مجموعة ذات سعرات أقل وأخرى ذات سعرات أعلى) بدلاً من مركز واحد متماثل.


## Running Time vs Calories by Gender

### English

We revisit the relationship between `Running Time(min)` and `Calories Burned`, this time coloring the points by `Gender` to see if the pattern differs between groups.

### العربية

نعيد فحص العلاقة بين `Running Time(min)` و `Calories Burned`، ولكن هذه المرة مع تلوين النقاط حسب `Gender` لمعرفة ما إذا كان النمط يختلف بين الفئتين.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Running Time(min)",
    y="Calories Burned",
    hue="Gender",
    alpha=0.6
)

plt.title("Running Time vs Calories Burned by Gender")
plt.xlabel("Running Time(min)")
plt.ylabel("Calories Burned")

save_and_display_figure("scatter_gender_running_time_calories.png")


### Interpretation

**English:**
Compare the spread and general trend of the Male-colored points against the Female-colored points. If the two colors overlap heavily across the same region of the plot with a similarly unclear or similarly weak trend, the relationship does not appear to differ meaningfully between gender groups in this dataset. If one color shows a visibly different pattern, this is worth noting as an observed (not causal) difference.

**العربي:**
قارن انتشار النقاط الخاصة بالذكور مع النقاط الخاصة بالإناث والاتجاه العام لكل منها. إذا تداخل اللونان بشكل كبير في نفس منطقة الرسم مع اتجاه غير واضح أو ضعيف بشكل مماثل، فإن العلاقة لا تبدو مختلفة بشكل ملحوظ بين فئتي الجنس في هذه البيانات. أما إذا أظهر أحد اللونين نمطًا مختلفًا بشكل واضح، فهذا يستحق الإشارة إليه كفارق ملحوظ (وليس سببيًا).


## Visualization Summary

### English

- **Target distribution:** `Calories Burned` was visualized with a histogram/KDE and a standalone KDE plot; refer to the printed shape and any skewness observed in those cells rather than assuming a specific shape here.
- **Feature distributions:** each numerical feature (Age, Height, Weight, BMI, Running Time, Running Speed, Distance, Average Heart Rate) was plotted individually; check each saved figure for its actual concentration, spread, and shape.
- **Outliers:** boxplots were generated for every numerical feature; any points beyond the whiskers in those figures represent the potential outliers actually present in the data.
- **Relationships with the target:** scatter plots and Pearson correlation coefficients were computed for Running Time, Distance, Average Heart Rate, Weight, and Age against `Calories Burned` — refer to the printed correlation values in each corresponding cell for the exact strength and direction.
- **Feature-to-feature relationships:** Running Time vs Distance and Running Speed vs Distance were examined with scatter plots and Pearson correlations.
- **Correlation structure:** the full correlation heatmap and the sorted target-correlation bar chart summarize which features are most and least linearly related to `Calories Burned`.
- **Gender-related observations:** the dataset contains a comparison of gender group sizes (count plot), Calories Burned by Gender (boxplot and violin plot), and a gender-colored scatter plot of Running Time vs Calories Burned.
- **Possible useful features for Machine Learning:** the features with the largest absolute correlation with `Calories Burned` in the sorted correlation table (Cell 20) are the first candidates worth considering for the modeling stage; features with correlations near zero may contribute less linear predictive value on their own, though they could still be useful in a non-linear model.
- **Limitations:** correlation analysis only captures linear relationships and does not imply causation. Non-linear relationships between features and the target, if present, would not necessarily show up as a strong Pearson correlation and would require additional modeling techniques to detect.

### العربية

- **توزيع المتغير المستهدف:** تم تصور `Calories Burned` باستخدام مخطط تكراري مع منحنى كثافة، ومخطط كثافة مستقل؛ يُرجى الرجوع إلى الشكل المطبوع وأي انحراف ملحوظ في تلك الخلايا بدلاً من افتراض شكل معين هنا.
- **توزيعات الخصائص:** تم رسم كل خاصية رقمية (العمر، الطول، الوزن، مؤشر كتلة الجسم، زمن الجري، سرعة الجري، المسافة، متوسط معدل ضربات القلب) بشكل منفصل؛ يُرجى فحص كل رسم محفوظ لمعرفة تركزه وانتشاره وشكله الفعلي.
- **القيم المتطرفة:** تم إنشاء مخططات صندوق لكل خاصية رقمية؛ أي نقاط تظهر خارج الشعيرات في تلك الرسومات تمثل القيم المتطرفة المحتملة الموجودة فعليًا في البيانات.
- **العلاقات مع المتغير المستهدف:** تم إنشاء مخططات انتشار وحساب معاملات ارتباط بيرسون لكل من زمن الجري والمسافة ومتوسط معدل ضربات القلب والوزن والعمر مقابل `Calories Burned` — يُرجى الرجوع إلى قيم الارتباط المطبوعة في كل خلية مقابلة لمعرفة القوة والاتجاه الدقيقين.
- **العلاقات بين الخصائص نفسها:** تم فحص العلاقة بين زمن الجري والمسافة، وبين سرعة الجري والمسافة، باستخدام مخططات انتشار ومعاملات ارتباط بيرسون.
- **بنية الارتباط:** توفر الخريطة الحرارية الكاملة للارتباط والمخطط الشريطي المرتب لارتباطات المتغير المستهدف ملخصًا للخصائص الأكثر والأقل ارتباطًا خطيًا بـ `Calories Burned`.
- **الملاحظات المتعلقة بالجنس:** تحتوي البيانات على مقارنة لأحجام فئتي الجنس (مخطط عد)، وتوزيع `Calories Burned` حسب الجنس (مخطط صندوق ومخطط كمان)، ومخطط انتشار ملون حسب الجنس لزمن الجري مقابل السعرات المحروقة.
- **الخصائص المحتملة المفيدة للتعلم الآلي:** الخصائص ذات أعلى قيمة مطلقة للارتباط مع `Calories Burned` في جدول الارتباط المرتب (الخلية 20) هي أول المرشحين الجديرين بالاعتبار لمرحلة النمذجة؛ أما الخصائص ذات الارتباط القريب من الصفر فقد تساهم بقيمة تنبؤية خطية أقل بمفردها، رغم أنها قد تظل مفيدة في نموذج غير خطي.
- **القيود:** يعكس تحليل الارتباط العلاقات الخطية فقط ولا يعني السببية. العلاقات غير الخطية بين الخصائص والمتغير المستهدف، إن وُجدت، لن تظهر بالضرورة كارتباط بيرسون قوي، وستتطلب تقنيات نمذجة إضافية لاكتشافها.


## Generated Figures

### English

Finally, we list every PNG file that currently exists inside `reports/figures` to verify that all visualizations produced by this notebook were actually saved to disk.

### العربية

أخيرًا، نقوم بإدراج كل ملف PNG موجود حاليًا داخل مجلد `reports/figures` للتأكد من أن جميع الرسومات التي تم إنشاؤها في هذا الدفتر قد تم حفظها فعليًا على القرص.


In [ ]:
figure_files = sorted(FIGURES_DIR.glob("*.png"))

figures_summary = pd.DataFrame({
    "Filename": [f.name for f in figure_files],
    "Path": [str(f) for f in figure_files],
    "File Size (KB)": [round(f.stat().st_size / 1024, 2) for f in figure_files],
})

print(f"Total figures found: {len(figures_summary)}")
figures_summary
